Let's get our data cleaned

In [62]:
import pandas as pd

In [63]:
# To get just the Medal Counts description part:
medal_desc = pd.read_csv('2025_Problem_C_Data/data_dictionary.csv', skiprows=1, nrows=7, encoding = 'latin1')

# To get the Hosts description:
# You'd count the rows from the top to where "summerOly_hosts.csv" starts
hosts_desc = pd.read_csv('2025_Problem_C_Data/data_dictionary.csv', skiprows=12, nrows=3, encoding = 'latin1')
athletes = pd.read_csv("2025_Problem_C_Data/summerOly_athletes.csv")
hosts = pd.read_csv("2025_Problem_C_Data/summerOly_hosts.csv")
medal_counts = pd.read_csv("2025_Problem_C_Data/summerOly_medal_counts.csv")
# 1. Use encoding='latin1' or 'cp1252' to handle the '' characters
# 2. Use nrows to stop reading before the "Data set of current..." paragraph begins
df_programs = pd.read_csv('2025_Problem_C_Data/summerOly_programs.csv', 
    encoding='latin1', 
    nrows=75  # Adjust this number to the last row of actual data
    )

In [64]:
history = medal_counts.merge(hosts, on="Year", how="left")
athlete_counts = athletes.groupby(['Year', 'Team']).size().reset_index(name='Total_Athletes')
medalist_filter = athletes[athletes['Medal'] != 'No medal']
medalist_counts = medalist_filter.groupby(['Year', 'Team']).size().reset_index(name='Medal_Winning_Athletes')

# 4. Merge these two summaries together
athlete_features = athlete_counts.merge(medalist_counts, on=['Year', 'Team'], how='left')

# 5. Calculate Efficiency (Percentage of athletes who won a medal)
athlete_features['Athlete_Efficiency'] = athlete_features['Medal_Winning_Athletes'] / athlete_features['Total_Athletes']

In [65]:
# Assuming 'history' is the table you made by joining medal_counts and hosts
final_training_data = history.merge(athlete_features, left_on=['Year', 'NOC'], right_on = ['Year', 'Team'], how='left')
final_training_data = final_training_data.drop(columns=['Team'])
final_training_data = final_training_data.dropna(subset=['Total_Athletes'])
final_training_data.head(15)

,Rank,NOC,Gold,Silver,Bronze,Total,Year,Host,Total_Athletes,Medal_Winning_Athletes,Athlete_Efficiency
0,1,United States,11,7,2,20,1896,"Athens, Greece",27.0,20.0,0.740741
1,2,Greece,10,18,19,47,1896,"Athens, Greece",140.0,44.0,0.314286
2,3,Germany,6,5,2,13,1896,"Athens, Greece",93.0,31.0,0.333333
3,4,France,5,4,2,11,1896,"Athens, Greece",26.0,11.0,0.423077
4,5,Great Britain,2,3,2,7,1896,"Athens, Greece",23.0,7.0,0.304348
5,6,Hungary,2,1,3,6,1896,"Athens, Greece",18.0,6.0,0.333333
6,7,Austria,2,1,2,5,1896,"Athens, Greece",8.0,5.0,0.625000
7,8,Australia,2,0,0,2,1896,"Athens, Greece",4.0,2.0,0.500000
8,9,Denmark,1,2,3,6,1896,"Athens, Greece",15.0,6.0,0.400000
9,10,Switzerland,1,2,0,3,1896,"Athens, Greece",8.0,3.0,0.375000


We have a bunch of individual athletes who are not apart of a country. We are going to drop them.